# 第 4 章 · 多智能体与工作流（ADK 的杀手锏）

> 单 Agent 的天花板很明显：prompt 越写越长、职责互相干扰、无法并行。**多智能体（Multi-Agent）** 的思路是"分而治之"——像组建公司一样组建 Agent 团队。ADK 提供两种协作范式：
> 1. **LLM 驱动的委派**：父 Agent 自主决定把任务转交给哪个 `sub_agent`（灵活）；
> 2. **工作流 Agent**：`Sequential / Parallel / Loop` 三种确定性编排（可靠）。
>
> 本章把两种范式各做一个完整项目，并对比它们的适用场景。

---

## 1. 多智能体系统的总图

```mermaid
flowchart TB
    subgraph PATTERN1["范式一：LLM 驱动委派（智能路由）"]
        R["🧭 Root Agent<br/>读 description 做决策"]
        R -.转移.-> SA["子 Agent A"]
        R -.转移.-> SB["子 Agent B"]
    end
    subgraph PATTERN2["范式二：工作流编排（确定性管道）"]
        SEQ["⛓️ SequentialAgent<br/>串行流水线"]
        PAR["🌿 ParallelAgent<br/>并行扇出"]
        LOOP["🔁 LoopAgent<br/>迭代循环"]
    end
    style R fill:#e8f0fe,stroke:#4285f4,stroke-width:2px
    style SEQ fill:#e6f4ea,stroke:#34a853
    style PAR fill:#fef7e0,stroke:#fbbc04
    style LOOP fill:#fce8e6,stroke:#ea4335
```

| 范式 | 控制者 | 优点 | 适用 |
|---|---|---|---|
| LLM 委派 | 模型自主决策 | 灵活、能处理开放意图 | 客服路由、意图分发 |
| Sequential | 开发者写死顺序 | 可预期、易调试 | 流水线加工（写-改-审） |
| Parallel | 同时扇出 | 省时间 | 多角度分析、批量处理 |
| Loop | 循环直到满足条件 | 自我修正 | 迭代优化、质量打磨 |

> 📌 **版本前沿**：ADK 2.6+ 已把 `SequentialAgent / ParallelAgent / LoopAgent` 标记为 **deprecated**，官方正在演进为图编排式的 `google.adk.workflow.Workflow`（按"构建图 → 调度节点 → 汇聚结果"的方式工作，目前尚不能嵌套为 LlmAgent 的子 Agent）。这释放了一个强烈信号：**ADK 的编排模型正在向 LangGraph 式的图模型靠拢**。本章仍使用工作流 Agent 教学——它们在当前版本完全可用（运行时的 DeprecationWarning 即来源于此），且这些概念会原样平移到新 API。

先准备本章通用的运行辅助函数（升级版：会显示每个事件的**作者**，方便观察 Agent 间的接力）：


In [1]:
import os
assert os.environ.get("DEEPSEEK_API_KEY"), "请先设置 DEEPSEEK_API_KEY"

from google.adk.agents import Agent
from google.adk.models.lite_llm import LiteLlm
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

APP, USER = "adk_ch04", "student"
DS = lambda: LiteLlm(model="deepseek/deepseek-chat")  # 每个 Agent 独立模型实例

async def run_team(root_agent, query, session_id="demo", show_state_keys=None):
    """运行一个（可能是多智能体的）Agent 系统，打印接力过程，返回 session。"""
    ss = InMemorySessionService()
    await ss.create_session(app_name=APP, user_id=USER, session_id=session_id)
    runner = Runner(agent=root_agent, app_name=APP, session_service=ss)
    msg = types.Content(role="user", parts=[types.Part(text=query)])
    print(f"🧑 {query}\n" + "═" * 60)
    async for ev in runner.run_async(user_id=USER, session_id=session_id, new_message=msg):
        if not (ev.content and ev.content.parts):
            continue
        for p in ev.content.parts:
            if p.function_call:
                print(f"  ⚡ [{ev.author}] 调用 {p.function_call.name}({dict(p.function_call.args)})")
            elif p.function_response:
                print(f"  📦 [{ev.author}] {p.function_response.name} 返回（略）")
            elif p.text:
                tag = "🏁 最终" if ev.is_final_response() else "🗣️ 中间"
                print(f"  {tag} [{ev.author}] {p.text[:120]}")
    session = await ss.get_session(app_name=APP, user_id=USER, session_id=session_id)
    if show_state_keys:
        print("═" * 60, "\n🗂️ Session 状态快照：")
        for k in show_state_keys:
            v = session.state.get(k, "（无）")
            print(f"  {k} = {str(v)[:100]}")
    return session

print("✅ 环境就绪")


✅ 环境就绪


---

## 2. 范式二 · SequentialAgent：串行流水线

**场景**：内容生产流水线——`大纲 → 初稿 → 润色`，每一步由专职 Agent 完成，上一步的产出通过 `output_key` 存入状态，下一步用 `{key}` 插值读取。

```mermaid
flowchart LR
    U["主题输入"] --> A["1️⃣ 大纲师<br/>output_key=outline"]
    A -->|"state['outline']"| B["2️⃣ 撰稿人<br/>读{outline}<br/>output_key=draft"]
    B -->|"state['draft']"| C["3️⃣ 润色师<br/>读{draft}<br/>output_key=final"]
    C --> O["成品文案"]
    style A fill:#e8f0fe,stroke:#4285f4
    style B fill:#e6f4ea,stroke:#34a853
    style C fill:#fef7e0,stroke:#fbbc04
```

这就是第 2 章 `output_key` + `{state插值}` 的实战组合：


In [2]:
from google.adk.agents import SequentialAgent

outliner = Agent(
    name="outliner", model=DS(),
    instruction="你是写作大纲师。根据用户主题列出 3 个要点的大纲，每点一句话。只输出大纲。",
    description="生成文章大纲",
    output_key="outline",
)

writer = Agent(
    name="writer", model=DS(),
    instruction="你是撰稿人。根据下面的大纲写一段 150 字左右的短文：\n{outline}\n只输出正文。",
    description="根据大纲写初稿",
    output_key="draft",
)

polisher = Agent(
    name="polisher", model=DS(),
    instruction="你是润色师。把下面的初稿改得更生动（保持字数，加入一个比喻）：\n{draft}\n只输出润色后的成稿。",
    description="润色初稿成成品",
    output_key="final",
)

pipeline = SequentialAgent(
    name="writing_pipeline",
    sub_agents=[outliner, writer, polisher],  # 按列表顺序依次执行
)

session = await run_team(pipeline, "主题：程序员为什么需要学习写作", session_id="seq",
                         show_state_keys=["outline", "final"])


🧑 主题：程序员为什么需要学习写作
════════════════════════════════════════════════════════════


C:\Users\37245\AppData\Local\Temp\ipykernel_42276\3322429214.py:24: DeprecationWarning: SequentialAgent is deprecated in favor of Workflow and will be removed in a future version. Workflow cannot yet be used as an LlmAgent sub-agent.
  pipeline = SequentialAgent(


23:09:32 - LiteLLM:WARNING: get_model_cost_map.py:289 - LiteLLM: Failed to fetch remote model cost map from https://raw.githubusercontent.com/BerriAI/litellm/main/model_prices_and_context_window.json: The read operation timed out. Falling back to local backup.


  🏁 最终 [outliner] 1. 写作能帮助程序员理清技术思路，提升系统设计和问题解决的逻辑性。  
2. 通过文档、博客或API说明的写作，程序员能更高效地传递代码意图，促进团队协作与知识沉淀。  
3. 持续写作可塑造个人技术品牌，增强职业竞争力，并促使深度复盘与


  🏁 最终 [writer] 写作对程序员而言，绝非可有可无的附加技能，而是“第二大脑”的锻造过程。当代码逻辑在脑海中混沌时，迫使自己落笔梳理，往往能暴露出设计中的隐藏缺陷，从而提升系统架构的严谨性与问题解决的条理性。同时，一份清晰的文档或博客，是代码意图的最佳翻译官，


  🏁 最终 [polisher] 写作对程序员而言，绝非可有可无的附加技能，而是“第二大脑”的锻造过程。当代码逻辑在脑海中混沌时，迫使自己落笔梳理，往往能像在迷雾中点亮一盏灯，暴露出设计中的隐藏缺陷，从而提升系统架构的严谨性与问题解决的条理性。同时，一份清晰的文档或博客，是
════════════════════════════════════════════════════════════ 
🗂️ Session 状态快照：
  outline = 1. 写作能帮助程序员理清技术思路，提升系统设计和问题解决的逻辑性。  
2. 通过文档、博客或API说明的写作，程序员能更高效地传递代码意图，促进团队协作与知识沉淀。  
3. 持续写作可塑造个人技
  final = 写作对程序员而言，绝非可有可无的附加技能，而是“第二大脑”的锻造过程。当代码逻辑在脑海中混沌时，迫使自己落笔梳理，往往能像在迷雾中点亮一盏灯，暴露出设计中的隐藏缺陷，从而提升系统架构的严谨性与问题解决


> 🔍 观察输出中每个事件的作者：`outliner → writer → polisher` 严格按序接力，状态在三人之间无缝传递。**整个过程没有任何 LLM 路由决策**——这就是"确定性编排"的可靠感。

---

## 3. 范式二 · ParallelAgent：并行扇出

**场景**：商业分析——同一命题，**技术、市场、风险**三个视角同时开分析会，最后由汇总员合并结论。

```mermaid
flowchart TD
    U["命题输入"] --> P{"ParallelAgent<br/>同时启动"}
    P --> T["🔬 技术分析师<br/>output_key=tech"]
    P --> M["📈 市场分析师<br/>output_key=market"]
    P --> K["⚠️ 风险分析师<br/>output_key=risk"]
    T --> S["🧾 汇总员<br/>读三个 key 合并"]
    M --> S
    K --> S
    style P fill:#fef7e0,stroke:#fbbc04,stroke-width:2px
```

> ⚠️ **关键规则**：并行分支共享同一份 Session 状态，所以每个分支必须写**不同的** `output_key`，否则会互相覆盖。

并行 + 串行可以**自由嵌套**——这正是 ADK 工作流的组合美学：


In [3]:
from google.adk.agents import ParallelAgent

def make_analyst(name, angle, key):
    return Agent(
        name=name, model=DS(),
        instruction=f"你是{angle}分析师。只从{angle}视角分析用户给的命题，80 字以内，直入主题。",
        description=f"{angle}视角分析",
        output_key=key,
    )

brainstorm = ParallelAgent(
    name="brainstorm",
    sub_agents=[
        make_analyst("tech_analyst", "技术可行性", "tech"),
        make_analyst("market_analyst", "市场前景", "market"),
        make_analyst("risk_analyst", "潜在风险", "risk"),
    ],
)

merger = Agent(
    name="merger", model=DS(),
    instruction="""你是首席顾问。综合三位分析师的意见给出一段总结与一句行动建议：
【技术】{tech}
【市场】{market}
【风险】{risk}""",
    description="汇总多视角分析",
    output_key="report",
)

# Sequential 套 Parallel：先并行扇出，再串行汇总
think_tank = SequentialAgent(name="think_tank", sub_agents=[brainstorm, merger])

session = await run_team(think_tank, "命题：在高校推广 AI 编程助教", session_id="par",
                         show_state_keys=["report"])


🧑 命题：在高校推广 AI 编程助教
════════════════════════════════════════════════════════════


C:\Users\37245\AppData\Local\Temp\ipykernel_42276\1853445323.py:11: DeprecationWarning: ParallelAgent is deprecated in favor of Workflow and will be removed in a future version. Workflow cannot yet be used as an LlmAgent sub-agent.
  brainstorm = ParallelAgent(
C:\Users\37245\AppData\Local\Temp\ipykernel_42276\1853445323.py:31: DeprecationWarning: SequentialAgent is deprecated in favor of Workflow and will be removed in a future version. Workflow cannot yet be used as an LlmAgent sub-agent.
  think_tank = SequentialAgent(name="think_tank", sub_agents=[brainstorm, merger])


  🏁 最终 [tech_analyst] 技术可行。AI编程助教可自动批改、答疑、生成练习，高校现有软硬件基本支持。但需注意模型准确率、数据隐私及与教学体系兼容性。总体可行性高，需试点验证。


  🏁 最终 [risk_analyst] 潜在风险：过度依赖AI导致学生编程基础薄弱，抑制独立解决问题的能力和创新思维；学术诚信可能因自动生成代码而受到挑战；教师评估与实际技能掌握脱节。
  🏁 最终 [market_analyst] 高校市场渗透率低，教师付费意愿弱，但政策利好与就业刚需推动潜力大。短期靠试点补贴，长期看校企合作与课程绑定，利润空间有限，生态价值更突出。


  🏁 最终 [merger] **综合总结：**  
三位分析师从不同维度评估了“高校推广AI编程助教”这一命题，呈现出高度互补的结论：**技术上可行且基础设施基本具备，市场潜力存在但短期变现困难，同时伴随不容忽视的教学质量与学术诚信风险**。三方共识指向一个核心判断—
════════════════════════════════════════════════════════════ 
🗂️ Session 状态快照：
  report = **综合总结：**  
三位分析师从不同维度评估了“高校推广AI编程助教”这一命题，呈现出高度互补的结论：**技术上可行且基础设施基本具备，市场潜力存在但短期变现困难，同时伴随不容忽视的教学质量与学术


---

## 4. 范式二 · LoopAgent：迭代打磨循环

**场景**：文案打磨——`写手` 与 `毒舌评审` 反复过招，直到评审满意为止。循环的**退出机制**有两种：

1. `max_iterations`：兜底上限，防止死循环；
2. **主动退出**：某个 Agent 调用一个设置 `tool_context.actions.escalate = True` 的工具，循环立即终止。

```mermaid
flowchart LR
    W["✍️ 写手<br/>写/改文案"] --> C["🧐 评审<br/>打分"]
    C -->|"不满意：给意见"| W
    C -->|"满意：调用 exit_loop"| E(["✅ 退出循环"])
    L["LoopAgent<br/>max_iterations=4 兜底"] -.包裹.-> W
    style L fill:#fce8e6,stroke:#ea4335,stroke-width:2px
```


In [4]:
from google.adk.agents import LoopAgent
from google.adk.tools import ToolContext

def exit_loop(tool_context: ToolContext) -> dict:
    """当文案质量已经达标时调用此工具，立即结束迭代循环。"""
    tool_context.actions.escalate = True   # 向 LoopAgent 发出"收工"信号
    return {"status": "评审通过，循环结束"}

copywriter = Agent(
    name="copywriter", model=DS(),
    instruction="你是广告写手。根据主题写一句 slogan；如果会话中已有评审意见，必须针对意见修改后再提交。只输出 slogan 本身。",
    description="撰写/修改 slogan",
    output_key="slogan",
)

critic = Agent(
    name="critic", model=DS(),
    instruction="""你是毒舌评审。审查当前 slogan：{slogan}
- 如果朗朗上口且有记忆点 → 直接调用 exit_loop 工具，不要输出多余文字；
- 否则 → 用一句话指出问题（存入你的回复），供写手下一轮修改。""",
    description="评审 slogan 并决定是否收工",
    tools=[exit_loop],
)

polish_loop = LoopAgent(
    name="polish_loop",
    sub_agents=[copywriter, critic],  # 每一轮：先写后评
    max_iterations=4,                 # 兜底：最多 4 轮
)

session = await run_team(polish_loop, "主题：无糖气泡水，主打『0 负担的快乐』", session_id="loop",
                         show_state_keys=["slogan"])


🧑 主题：无糖气泡水，主打『0 负担的快乐』
════════════════════════════════════════════════════════════


C:\Users\37245\AppData\Local\Temp\ipykernel_42276\1897030076.py:25: DeprecationWarning: LoopAgent is deprecated in favor of Workflow and will be removed in a future version. Workflow cannot yet be used as an LlmAgent sub-agent.
  polish_loop = LoopAgent(


  🏁 最终 [copywriter] 0 糖0压，一口快乐不设限。


D:\Python\Lib\site-packages\google\adk\models\llm_request.py:273: UserWarning: [EXPERIMENTAL] feature FeatureName.JSON_SCHEMA_FOR_FUNC_DECL is enabled.
  declaration = tool._get_declaration()


  ⚡ [critic] 调用 exit_loop({})
  📦 [critic] exit_loop 返回（略）


  🏁 最终 [critic] slogan 已通过评审，循环结束。
════════════════════════════════════════════════════════════ 
🗂️ Session 状态快照：
  slogan = 0 糖0压，一口快乐不设限。


> 🔍 你可能看到 1~4 轮不等的"写 → 评"往复——循环次数由**评审的判断**动态决定，这就是 LoopAgent 的"自我修正"能力。

---

## 5. 范式一 · LLM 驱动委派：智能路由

工作流 Agent 的路线是**开发者画好的**；而委派模式把路由决策交给 **LLM**：

- 父 Agent 声明 `sub_agents=[...]` 后，框架自动给它注入一个 `transfer_to_agent` 工具；
- 父 Agent 阅读各子 Agent 的 **`description`**，像调度员一样决定"这单给谁"；
- 转移后，**子 Agent 接管对话**（它能看到完整历史），用户后续消息也直接由它处理。

**场景**：校园服务台——前台 Agent 把问题分流给"教务专家"或"IT 支持"。

```mermaid
flowchart TD
    U["🧑 学生提问"] --> F["🧭 前台 Agent<br/>（不答题，只分流）"]
    F -->|"transfer_to_agent<br/>（读 description 决策）"| A["📚 教务专家<br/>课表/考试/学分"]
    F -->|"transfer_to_agent"| B["💻 IT 支持<br/>账号/网络/设备"]
    A --> U
    B --> U
    style F fill:#e8f0fe,stroke:#4285f4,stroke-width:2px
```


In [5]:
academic = Agent(
    name="academic_affairs", model=DS(),
    instruction="你是教务处专家，用中文简洁回答课表、考试、学分相关问题。",
    description="负责教务问题：课程表、考试安排、学分与成绩政策。",  # ← 路由依据！
)

it_support = Agent(
    name="it_support", model=DS(),
    instruction="你是校园 IT 支持，用中文给出账号、网络、设备问题的排查步骤，分点说明。",
    description="负责 IT 问题：校园网、账号登录、邮箱与电脑故障。",   # ← 路由依据！
)

front_desk = Agent(
    name="front_desk", model=DS(),
    instruction="你是服务台前台，自己不回答问题，必须根据问题类型把对话转交给对应专家。",
    description="校园服务台总机",
    sub_agents=[academic, it_support],   # ← 声明可委派的对象
)

await run_team(front_desk, "你好，我连不上校园网了，而且想问问下周的高数考试安排。", session_id="route")


🧑 你好，我连不上校园网了，而且想问问下周的高数考试安排。
════════════════════════════════════════════════════════════


  🗣️ 中间 [front_desk] I need to transfer both queries to the appropriate agents. The IT issue (campus network connection) goes to it_support, 
  ⚡ [front_desk] 调用 transfer_to_agent({'agent_name': 'it_support'})
  ⚡ [front_desk] 调用 transfer_to_agent({'agent_name': 'academic_affairs'})
  📦 [front_desk] transfer_to_agent 返回（略）
  📦 [front_desk] transfer_to_agent 返回（略）


  🗣️ 中间 [academic_affairs] 用户同时提出了两个问题：校园网连接问题和考试安排问题。

校园网连接问题属于IT问题，应该转给 it_support 处理。考试安排问题属于教务问题，我可以回答。

不过用户的问题被转到了我这里，其中包含了IT问题和教务问题。对于IT问题，
  ⚡ [academic_affairs] 调用 transfer_to_agent({'agent_name': 'it_support'})
  📦 [academic_affairs] transfer_to_agent 返回（略）


  🗣️ 中间 [it_support] 你好！我是校园 IT 支持，先帮你解决校园网连接的问题。

关于你提到的**高数考试安排**，这属于教务问题，我应该把这个问题转给负责教务的同事来回答。

我先帮你处理校园网的排查，同时把考试安排问题转给教务专员。

以下是校园网连接问题的
  ⚡ [it_support] 调用 transfer_to_agent({'agent_name': 'academic_affairs'})
  📦 [it_support] transfer_to_agent 返回（略）


  🏁 最终 [academic_affairs] 你好！我是教务专员，关于**下周的高数考试安排**，我来为你解答。

目前我这边需要确认一下你的具体情况才能提供准确的考试信息：

1. **你的专业/班级**：不同专业的高数课程代码和考试时间可能不同。
2. **高数类型**：是高等数学


Session(id='route', app_name='adk_ch04', user_id='student', state={}, events=[Event(model_version=None, content=Content(
  parts=[
    Part(
      text='你好，我连不上校园网了，而且想问问下周的高数考试安排。'
    ),
  ],
  role='user'
), grounding_metadata=None, partial=None, turn_complete=None, turn_complete_reason=None, finish_reason=None, error_code=None, error_message=None, interrupted=None, custom_metadata=None, usage_metadata=None, live_session_resumption_update=None, live_session_id=None, go_away=None, voice_activity=None, input_transcription=None, output_transcription=None, avg_logprobs=None, logprobs_result=None, cache_metadata=None, citation_metadata=None, interaction_id=None, environment_id=None, invocation_id='e-0d56e8f2-1393-4a28-b1c8-e89df9e84c4e', author='user', actions=EventActions(skip_summarization=None, state_delta={}, artifact_delta={}, transfer_to_agent=None, escalate=None, requested_auth_configs={}, requested_tool_confirmations={}, compaction=None, end_of_agent=None, agent_state=None, rewin

> 🔍 观察事件流里的 `transfer_to_agent` 调用——LLM 自己读了 description、自己做了（甚至可能两次）路由决策。**注意**：这个例子里用户的复合需求暴露了委派模式的一个特点——转移是"一对一接管"，复杂需求可能需要多次转移，或在设计时拆成多轮。

---

## 6. 两种范式如何选择？

| 考量 | LLM 委派 | 工作流编排 |
|---|---|---|
| 路线是否已知/固定 | 未知、开放意图 | 已知、步骤明确 |
| 可靠性要求 | 中（模型可能选错） | 高（100% 按图执行） |
| token 开销 | 高（每步都要模型决策） | 低 |
| 延迟 | 高（多一次 LLM 决策） | 低（Parallel 还能提速） |
| 典型场景 | 客服分流、开放问答 | 内容生产、数据处理管道 |

> 💡 **工程实践**：真实系统通常是**混合架构**——顶层用 LLM 委派做意图路由，每个子 Agent 内部用 Sequential/Loop 做确定性加工。ADK 允许任意嵌套：工作流 Agent 可以作为 `sub_agents` 被委派，LLM Agent 也可以作为工作流的一环。

---

## 7. 与 LangGraph 对照 🔄

| ADK | LangGraph 等价物 | 差异 |
|---|---|---|
| `SequentialAgent` | 一条直线边链 `A→B→C` | LangGraph 更啰嗦但可在任意节点插入逻辑 |
| `ParallelAgent` | 扇出边 + Reducer 合并状态 | LangGraph 需自己设计合并策略，自由度高 |
| `LoopAgent` + escalate | 条件边构成回边 `评→写` | 本质相同：循环即回边 |
| `transfer_to_agent` | Command/handoff 工具 + supervisor 模式 | ADK 内建；LangGraph 需自建或用 `langgraph-supervisor` |
| `output_key` + `{state}` 插值 | 节点返回 dict 更新 State | ADK 声明式；LangGraph 显式 |

**一句话**：ADK 把最常见的多智能体模式"预制"成了四种积木；LangGraph 给你图纸和零件，能造出任意形状，但每颗螺丝都要自己拧。（有趣的是：ADK 新一代 `Workflow` API 正在转向图编排——两大框架在"图"这个交汇点上越走越近。）

---

## 📌 本章要点回顾

- 多智能体两种范式：**LLM 委派**（灵活路由）与 **工作流 Agent**（确定性编排）；
- 工作流三剑客：`Sequential`（流水线）、`Parallel`（并行扇出+不同 output_key）、`Loop`（escalate 主动退出）；
- 委派模式靠 `sub_agents` + `description` 驱动，转移后子 Agent 接管对话；
- 真实系统 = 两种范式的**混合嵌套**。

> ➡️ 下一章：[05-会话-状态与记忆](05-会话-状态与记忆.ipynb) —— 深入 ADK 的三层记忆体系。
